## Understanding The Data

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anlgrbz/student-demographics-online-education-dataoulad")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\HP\.cache\kagglehub\datasets\anlgrbz\student-demographics-online-education-dataoulad\versions\1


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# تحميل جداول OULAD
# ============================================================
PATH = "C:/Users/HP/PycharmProjects/EduPredict/project II/OULAD/"
student_info    = pd.read_csv(PATH + "studentInfo.csv")
student_reg     = pd.read_csv(PATH + "studentRegistration.csv")
student_assess  = pd.read_csv(PATH + "studentAssessment.csv")
student_vle     = pd.read_csv(PATH + "studentVle.csv")
assessments     = pd.read_csv(PATH + "assessments.csv")
vle             = pd.read_csv(PATH + "vle.csv")
courses         = pd.read_csv(PATH + "courses.csv")

print("studentInfo:       ", student_info.shape)
print("studentRegistration:", student_reg.shape)
print("studentAssessment: ", student_assess.shape)
print("studentVle:        ", student_vle.shape)
print("assessments:       ", assessments.shape)
print("vle:               ", vle.shape)
print("courses:           ", courses.shape)

studentInfo:        (32593, 12)
studentRegistration: (32593, 5)
studentAssessment:  (173912, 5)
studentVle:         (10655280, 6)
assessments:        (206, 6)
vle:                (6364, 6)
courses:            (22, 3)


In [2]:
print("=== studentInfo ===")
print(student_info.head())
print("\n--- dtypes ---")
print(student_info.dtypes)
print("\n--- Null Values ---")
print(student_info.isnull().sum())
print("\n--- distributed final_result ---")
print(student_info['final_result'].value_counts())
print(student_info['final_result'].value_counts(normalize=True).round(3) * 100)

=== studentInfo ===
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%     55<=                     0   
1       HE Qualification   20-30%    35-55                     0   
2  A Level or Equivalent   30-40%    35-55                     0   
3  A Level or Equivalent   50-60%    35-55                     0   
4     Lower Than A Level   50-60%     0-35                     0   

   studied_credits disability final_result  
0              240          N         Pass  
1               60      

In [3]:
for name, df in [
    ("studentRegistration", student_reg),
    ("studentAssessment",   student_assess),
    ("studentVle",          student_vle),
    ("assessments",         assessments),
    ("vle",                 vle),
]:
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape}")
    print(df.head(3))
    print("Nulls:", df.isnull().sum().to_dict())
    print("-" * 50)


=== studentRegistration ===
Shape: (32593, 5)
  code_module code_presentation  id_student  date_registration  \
0         AAA             2013J       11391             -159.0   
1         AAA             2013J       28400              -53.0   
2         AAA             2013J       30268              -92.0   

   date_unregistration  
0                  NaN  
1                  NaN  
2                 12.0  
Nulls: {'code_module': 0, 'code_presentation': 0, 'id_student': 0, 'date_registration': 45, 'date_unregistration': 22521}
--------------------------------------------------

=== studentAssessment ===
Shape: (173912, 5)
   id_assessment  id_student  date_submitted  is_banked  score
0           1752       11391              18          0   78.0
1           1752       28400              22          0   70.0
2           1752       31604              17          0   72.0
Nulls: {'id_assessment': 0, 'id_student': 0, 'date_submitted': 0, 'is_banked': 0, 'score': 173}
---------------------

In [4]:
print("=== توزيع assessment_type ===")
print(assessments['assessment_type'].value_counts())

print("\n=== أنواع activity_type في VLE ===")
print(vle['activity_type'].value_counts())

print("\n=== date_registration — إحصاء (سلبي = قبل بداية المقرر) ===")
print(student_reg['date_registration'].describe())

print("\n=== كم طالب عنده أكثر من محاولة سابقة؟ ===")
print(student_info['num_of_prev_attempts'].value_counts())

print("\n=== studied_credits ===")
print(student_info['studied_credits'].describe())

=== توزيع assessment_type ===
TMA     106
CMA      76
Exam     24
Name: assessment_type, dtype: int64

=== أنواع activity_type في VLE ===
resource          2660
subpage           1055
oucontent          996
url                886
forumng            194
quiz               127
page               102
oucollaborate       82
questionnaire       61
ouwiki              49
dataplus            28
externalquiz        26
homepage            22
ouelluminate        21
glossary            21
dualpane            20
repeatactivity       5
htmlactivity         4
sharedsubpage        3
folder               2
Name: activity_type, dtype: int64

=== date_registration — إحصاء (سلبي = قبل بداية المقرر) ===
count    32548.000000
mean       -69.411300
std         49.260522
min       -322.000000
25%       -100.000000
50%        -57.000000
75%        -29.000000
max        167.000000
Name: date_registration, dtype: float64

=== كم طالب عنده أكثر من محاولة سابقة؟ ===
0    28421
1     3299
2      675
3      142
4  

## Feature Engineering

#### Target Variable

In [5]:
# Create binary target: 1 = at risk, 0 = not at risk
student_info['at_risk'] = student_info['final_result'].isin(['Fail', 'Withdrawn']).astype(int)

print(student_info['at_risk'].value_counts())
print(student_info['at_risk'].value_counts(normalize=True).round(3) * 100)

1    17208
0    15385
Name: at_risk, dtype: int64
1    52.8
0    47.2
Name: at_risk, dtype: float64


##  Group 1: Demographics from studentInfo


In [6]:
# --- ordinal mappings ---
age_map = {'0-35': 0, '35-55': 1, '55<=': 2}

edu_map = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}

# imd_band: extract lower bound as numeric, fill nulls with median
def parse_imd(val):
    if pd.isna(val):
        return np.nan
    try:
        return int(str(val).split('-')[0].replace('%','').strip())
    except:
        return np.nan

student_info['imd_numeric'] = student_info['imd_band'].apply(parse_imd)
imd_median = student_info['imd_numeric'].median()
student_info['imd_numeric'] = student_info['imd_numeric'].fillna(imd_median)

# Apply all mappings
student_info['age_numeric']  = student_info['age_band'].map(age_map)
student_info['edu_numeric']  = student_info['highest_education'].map(edu_map)
student_info['gender_bin']   = (student_info['gender'] == 'M').astype(int)
student_info['disability_bin'] = (student_info['disability'] == 'Y').astype(int)

# Verify
cols = ['age_numeric','edu_numeric','imd_numeric','gender_bin','disability_bin']
print(student_info[cols].describe().round(2))
print("\nNulls:", student_info[cols].isnull().sum().to_dict())

       age_numeric  edu_numeric  imd_numeric  gender_bin  disability_bin
count     32593.00     32593.00     32593.00    32593.00         32593.0
mean          0.30         1.74        42.04        0.55             0.1
std           0.47         0.75        27.68        0.50             0.3
min           0.00         0.00         0.00        0.00             0.0
25%           0.00         1.00        20.00        0.00             0.0
50%           0.00         2.00        40.00        1.00             0.0
75%           1.00         2.00        70.00        1.00             0.0
max           2.00         4.00        90.00        1.00             1.0

Nulls: {'age_numeric': 0, 'edu_numeric': 0, 'imd_numeric': 0, 'gender_bin': 0, 'disability_bin': 0}


## Group 2: Registration features

In [7]:
# Merge registration info onto student_info
reg = student_reg[['id_student','code_module','code_presentation',
                    'date_registration','date_unregistration']].copy()

df = student_info.merge(reg, on=['id_student','code_module','code_presentation'], how='left')

# days_to_start: positive = registered early (flip sign)
df['days_to_start'] = df['date_registration'] * -1

# registered_early: enrolled more than 30 days before start
df['registered_early'] = (df['days_to_start'] > 30).astype(int)

# Verify — NO unregistered feature (avoid leakage)
cols = ['days_to_start', 'registered_early']
print(df[cols].describe().round(2))
print("\nNulls:", df[cols].isnull().sum().to_dict())
print("\nregistered_early distribution:")
print(df['registered_early'].value_counts())

       days_to_start  registered_early
count       32548.00          32593.00
mean           69.41              0.72
std            49.26              0.45
min          -167.00              0.00
25%            29.00              0.00
50%            57.00              1.00
75%           100.00              1.00
max           322.00              1.00

Nulls: {'days_to_start': 45, 'registered_early': 0}

registered_early distribution:
1    23526
0     9067
Name: registered_early, dtype: int64


###  Group 3: Assessment features

In [8]:
# Merge assessments metadata with student scores
assess_full = student_assess.merge(
    assessments[['id_assessment','assessment_type','date','weight']],
    on='id_assessment', how='left'
)

# Fill missing scores with 0 (student did not submit)
assess_full['score'] = assess_full['score'].fillna(0)

# --- overall score features ---
assess_agg = assess_full.groupby('id_student').agg(
    avg_score          = ('score', 'mean'),
    num_submitted      = ('score', 'count'),
    num_failed         = ('score', lambda x: (x < 40).sum()),
).reset_index()

# submission rate: submitted vs total assessments per module
# total assessments per student (from assessments table via merge)
total_per_student = assess_full.groupby('id_student')['id_assessment'].nunique().reset_index()
total_per_student.columns = ['id_student', 'total_assessments']

assess_agg = assess_agg.merge(total_per_student, on='id_student', how='left')
assess_agg['submission_rate'] = (
    assess_agg['num_submitted'] / assess_agg['total_assessments']
).clip(0, 1)

# --- TMA and CMA scores separately ---
tma_scores = assess_full[assess_full['assessment_type'] == 'TMA'].groupby('id_student').agg(
    avg_tma_score = ('score', 'mean')
).reset_index()

cma_scores = assess_full[assess_full['assessment_type'] == 'CMA'].groupby('id_student').agg(
    avg_cma_score = ('score', 'mean')
).reset_index()

# --- submission timing: avg days late (positive = late) ---
timing = assess_full.dropna(subset=['date']).copy()
timing['days_late'] = timing['date_submitted'] - timing['date']
timing_agg = timing.groupby('id_student').agg(
    avg_days_late = ('days_late', 'mean')
).reset_index()

# Merge all assessment features together
assess_features = assess_agg \
    .merge(tma_scores,  on='id_student', how='left') \
    .merge(cma_scores,  on='id_student', how='left') \
    .merge(timing_agg,  on='id_student', how='left')

print(assess_features.shape)
print(assess_features.describe().round(2))
print("\nNulls:", assess_features.isnull().sum().to_dict())

(23369, 9)
       id_student  avg_score  num_submitted  num_failed  total_assessments  \
count    23369.00   23369.00       23369.00    23369.00           23369.00   
mean    709142.56      73.01           7.44        0.33               7.44   
std     555641.50      15.75           4.22        0.81               4.22   
min       6516.00       0.00           1.00        0.00               1.00   
25%     504602.00      65.00           4.00        0.00               4.00   
50%     589565.00      76.00           7.00        0.00               7.00   
75%     645147.00      84.30          11.00        0.00              11.00   
max    2698588.00     100.00          28.00       10.00              28.00   

       submission_rate  avg_tma_score  avg_cma_score  avg_days_late  
count          23369.0       22692.00       14400.00       23369.00  
mean               1.0          71.11          79.49         -12.49  
std                0.0          15.61          16.78          25.48  
min   

###  Group 4: VLE features

In [9]:
# Merge vle metadata to get activity_type per interaction
vle_full = student_vle.merge(
    vle[['id_site','activity_type']],
    on='id_site', how='left'
)

# --- overall engagement ---
vle_agg = vle_full.groupby('id_student').agg(
    total_clicks  = ('sum_click', 'sum'),
    active_days   = ('date', 'nunique'),
).reset_index()

vle_agg['avg_clicks_per_day'] = (
    vle_agg['total_clicks'] / vle_agg['active_days']
).round(2)

# --- clicks by activity type ---
def clicks_for_type(activity):
    return vle_full[vle_full['activity_type'] == activity] \
        .groupby('id_student')['sum_click'].sum().reset_index() \
        .rename(columns={'sum_click': f'{activity}_clicks'})

quiz_cl   = clicks_for_type('quiz')
forum_cl  = clicks_for_type('forumng')
resource_cl = clicks_for_type('resource')

# --- clicked before module start (date < 0) ---
early_click = vle_full[vle_full['date'] < 0] \
    .groupby('id_student')['sum_click'].sum().reset_index()
early_click.columns = ['id_student', 'early_clicks']
early_click['clicked_before_start'] = 1

# Merge all VLE features
vle_features = vle_agg \
    .merge(quiz_cl,     on='id_student', how='left') \
    .merge(forum_cl,    on='id_student', how='left') \
    .merge(resource_cl, on='id_student', how='left') \
    .merge(early_click[['id_student','clicked_before_start']], on='id_student', how='left')

# Fill nulls: students who never used that activity type
fill_cols = ['quiz_clicks','forumng_clicks','resource_clicks','clicked_before_start']
vle_features[fill_cols] = vle_features[fill_cols].fillna(0)

print(vle_features.shape)
print(vle_features.describe().round(2))
print("\nNulls:", vle_features.isnull().sum().to_dict())

(26074, 8)
       id_student  total_clicks  active_days  avg_clicks_per_day  quiz_clicks  \
count    26074.00      26074.00     26074.00            26074.00     26074.00   
mean    708745.33       1518.95        66.62               19.72       267.75   
std     552954.42       1935.99        56.63               11.77       511.35   
min       6516.00          1.00         1.00                1.00         0.00   
25%     506717.00        298.00        21.00               11.55         0.00   
50%     590115.00        824.00        52.00               17.00        79.00   
75%     645918.25       2018.00        99.00               25.38       271.00   
max    2698588.00      28615.00       286.00              157.71     13032.00   

       forumng_clicks  resource_clicks  clicked_before_start  
count        26074.00         26074.00              26074.00  
mean           305.80            42.58                  0.82  
std            655.42            80.04                  0.38  
min    

### Final Merge

In [10]:
# Start from the base dataframe (student_info + registration)
# Fix days_to_start nulls with median
days_median = df['days_to_start'].median()
df['days_to_start'] = df['days_to_start'].fillna(days_median)

# Select only the columns we need from base df
base_cols = [
    'id_student', 'code_module', 'code_presentation',
    'gender_bin', 'disability_bin', 'age_numeric', 'edu_numeric',
    'imd_numeric', 'num_of_prev_attempts', 'studied_credits',
    'days_to_start', 'registered_early',
    'at_risk'
]
final_df = df[base_cols].copy()

# Merge assessment features
final_df = final_df.merge(assess_features, on='id_student', how='left')

# Merge VLE features
final_df = final_df.merge(vle_features, on='id_student', how='left')

# Fix submission_rate properly:
# total_assessments per module-presentation (not per student)
module_assess_count = assessments.groupby(
    ['code_module','code_presentation']
)['id_assessment'].count().reset_index()
module_assess_count.columns = ['code_module','code_presentation','module_total_assessments']

final_df = final_df.merge(module_assess_count, on=['code_module','code_presentation'], how='left')
final_df['submission_rate'] = (
    final_df['num_submitted'].fillna(0) / final_df['module_total_assessments']
).clip(0, 1)

# Fill nulls for students with no assessment records
assess_fill = {
    'avg_score': 0, 'num_submitted': 0, 'num_failed': 0,
    'avg_tma_score': 0, 'avg_cma_score': 0,
    'avg_days_late': 0, 'total_assessments': 0
}
final_df.fillna(assess_fill, inplace=True)

# Fill nulls for students with no VLE records
vle_fill = {
    'total_clicks': 0, 'active_days': 0, 'avg_clicks_per_day': 0,
    'quiz_clicks': 0, 'forumng_clicks': 0, 'resource_clicks': 0,
    'clicked_before_start': 0
}
final_df.fillna(vle_fill, inplace=True)

# Drop helper columns not needed for modeling
final_df.drop(columns=['total_assessments'], inplace=True)

print("Shape:", final_df.shape)
print("\nNulls:")
print(final_df.isnull().sum()[final_df.isnull().sum() > 0])
print("\nSample:")
print(final_df.head(3))
print("\nat_risk distribution:")
print(final_df['at_risk'].value_counts())

Shape: (32593, 28)

Nulls:
Series([], dtype: int64)

Sample:
   id_student code_module code_presentation  gender_bin  disability_bin  \
0       11391         AAA             2013J           1               0   
1       28400         AAA             2013J           0               0   
2       30268         AAA             2013J           0               1   

   age_numeric  edu_numeric  imd_numeric  num_of_prev_attempts  \
0            2            3         90.0                     0   
1            1            3         20.0                     0   
2            1            2         30.0                     0   

   studied_credits  ...  avg_cma_score  avg_days_late  total_clicks  \
0              240  ...            0.0           -1.8         934.0   
1               60  ...            0.0            0.0        1435.0   
2               60  ...            0.0            0.0         281.0   

   active_days  avg_clicks_per_day  quiz_clicks  forumng_clicks  \
0         40.0       

### Save and split

In [11]:
from sklearn.model_selection import train_test_split
import os

# Create output directory
os.makedirs("edupredict_data", exist_ok=True)

# -------------------------------------------------------
# Step 1: Remove identifier/leakage columns before saving
# -------------------------------------------------------
# Keep code_module as a feature (different modules may have different patterns)
# Drop id_student, code_presentation (too specific, not generalizable)
model_df = final_df.drop(columns=['id_student', 'code_presentation']).copy()

# -------------------------------------------------------
# Step 2: Split — 80% train, 20% holdout test
# Stratify to preserve at_risk ratio in both splits
# -------------------------------------------------------
train_df, test_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df['at_risk']
)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print("\nTrain at_risk distribution:")
print(train_df['at_risk'].value_counts(normalize=True).round(3) * 100)
print("\nTest at_risk distribution:")
print(test_df['at_risk'].value_counts(normalize=True).round(3) * 100)

# -------------------------------------------------------
# Step 3: Save all versions
# -------------------------------------------------------
model_df.to_csv("edupredict_data/full_features.csv",  index=False)
train_df.to_csv("edupredict_data/train.csv",          index=False)
test_df.to_csv( "edupredict_data/test.csv",           index=False)

print("\nFiles saved.")

# -------------------------------------------------------
# Step 4: Verify files — reload and check
# -------------------------------------------------------
check_full  = pd.read_csv("edupredict_data/full_features.csv")
check_train = pd.read_csv("edupredict_data/train.csv")
check_test  = pd.read_csv("edupredict_data/test.csv")

print("\n--- Verification ---")
print(f"full_features : {check_full.shape}  — nulls: {check_full.isnull().sum().sum()}")
print(f"train         : {check_train.shape}  — nulls: {check_train.isnull().sum().sum()}")
print(f"test          : {check_test.shape}   — nulls: {check_test.isnull().sum().sum()}")
print(f"\nColumns match: {list(check_train.columns) == list(check_test.columns)}")
print(f"\nFeature columns ({len(model_df.columns)-1}):")
print([c for c in model_df.columns if c != 'at_risk'])

Train shape: (26074, 26)
Test shape:  (6519, 26)

Train at_risk distribution:
1    52.8
0    47.2
Name: at_risk, dtype: float64

Test at_risk distribution:
1    52.8
0    47.2
Name: at_risk, dtype: float64

Files saved.

--- Verification ---
full_features : (32593, 26)  — nulls: 0
train         : (26074, 26)  — nulls: 0
test          : (6519, 26)   — nulls: 0

Columns match: True

Feature columns (25):
['code_module', 'gender_bin', 'disability_bin', 'age_numeric', 'edu_numeric', 'imd_numeric', 'num_of_prev_attempts', 'studied_credits', 'days_to_start', 'registered_early', 'avg_score', 'num_submitted', 'num_failed', 'submission_rate', 'avg_tma_score', 'avg_cma_score', 'avg_days_late', 'total_clicks', 'active_days', 'avg_clicks_per_day', 'quiz_clicks', 'forumng_clicks', 'resource_clicks', 'clicked_before_start', 'module_total_assessments']


In [13]:

# EDUPREDICT — Feature Engineering Documentation
# =================================================
# Dataset : OULAD (Open University Learning Analytics Dataset)
# Students : 32,593 | Features : 25 | Target : at_risk (binary)
# Train    : 26,074 rows | Test (holdout) : 6,519 rows
# Split    : 80/20 stratified on at_risk

# -------------------------------------------------
# TARGET VARIABLE
# -------------------------------------------------
# at_risk = 1  →  final_result in [Fail, Withdrawn]  →  17,208 (52.8%)
# at_risk = 0  →  final_result in [Pass, Distinction] →  15,385 (47.2%)
# Note: unregistration date was NOT used as a feature (data leakage risk)

# -------------------------------------------------
# GROUP 1 — Demographics (source: studentInfo)
# -------------------------------------------------
# gender_bin            : 0=Female, 1=Male
# disability_bin        : 0=No, 1=Yes
# age_numeric           : 0=0-35, 1=35-55, 2=55+  (ordinal)
# edu_numeric           : 0=No Formal → 4=Post Graduate  (ordinal)
# imd_numeric           : deprivation index 0-100, nulls filled with median (40.0)
# num_of_prev_attempts  : how many times student retook this module
# studied_credits       : total credits student is studying this semester
# code_module           : module ID — kept because difficulty varies by module

# -------------------------------------------------
# GROUP 2 — Registration Behavior (source: studentRegistration)
# -------------------------------------------------
# days_to_start         : days between registration and module start (higher = earlier)
#                         nulls (45 rows) filled with median
# registered_early      : 1 if registered more than 30 days before start

# -------------------------------------------------
# GROUP 3 — Academic Performance (source: studentAssessment + assessments)
# -------------------------------------------------
# avg_score             : mean score across all submitted assessments (no-submit → 0)
# num_submitted         : number of assessments submitted
# num_failed            : number of assessments with score < 40
# submission_rate       : num_submitted / total assessments in module (0 to 1)
# avg_tma_score         : mean score on Tutor Marked Assessments only
# avg_cma_score         : mean score on Computer Marked Assessments only
# avg_days_late         : mean of (submission_date - due_date), positive = late

# -------------------------------------------------
# GROUP 4 — VLE Engagement (source: studentVle + vle)
# -------------------------------------------------
# total_clicks          : total interactions with VLE throughout the module
# active_days           : number of distinct days the student accessed VLE
# avg_clicks_per_day    : total_clicks / active_days
# quiz_clicks           : clicks on quiz-type activities
# forumng_clicks        : clicks on discussion forum activities
# resource_clicks       : clicks on resource/content pages
# clicked_before_start  : 1 if student interacted with VLE before module start date

# -------------------------------------------------
# DROPPED COLUMNS (reasons)
# -------------------------------------------------
# id_student        : identifier, not a feature
# code_presentation : historical (2013/2014 only), not generalizable
# final_result      : source of target, must not be included
# date_unregistration : direct leakage — only present for Withdrawn students

# -------------------------------------------------
# FILES SAVED
# -------------------------------------------------
# edupredict_data/full_features.csv  — all 32,593 rows
# edupredict_data/train.csv          — 26,074 rows for model training
# edupredict_data/test.csv           — 6,519 rows, locked until final evaluation
